# Creating the Vector Database
This notebook takes the baking knowledge base (Markdown file), implements a specific chunking strategy (3 total strategies tested), and stores these chunks in a vector similarity database using ChromaDB.

Note: This notebook only needs to be run once per chunking method. It creates the Chroma database for that configuration, which can then be reused without rebuilding it.

In other words,
1) Chunking: The raw baking corpus is segmented into chunks using one of 3 methods:
* recursive fixed-size text splitting
* markdown subheading splitting
* LangChain's Semantic chunking using Open-ai's embedding model (text-embedding-3-small)

2) Embedding: Each chunk is converted into an embedding using the selected embedding model (Google Generative AI embeddings)

3) Storage: The embeddings and their chunk metadata are stored in a ChromaDB collection. This enables similarity search during retrieval-augmented generation (RAG).

## Imports and Setup

All different chunking databases will be stored in the same directory called "chroma_db" (one database folder). But each different chunking method will be treated as a separate collection with a different name. 

In [25]:
#These are all the necessary Imports:
import os
import re
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter
)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.documents import Document

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_chroma import Chroma
import gradio as gr


In [2]:
# Load the API key from a local .env file (not included in the notebook for privacy reasons)
# Uses the free Google Gemini API key.
load_dotenv()

True

In [7]:
#DATA_PATH is location of the markdown file (Knowledge base) containing the baking corpus
DATA_PATH = r"data/baking_corpus.md" 
#The chroma database will be stored inside this project folder and will be called chroma_db
CHROMA_PATH = r"chroma_db"

#Extra test to check that the data file does exist (uploading the correct one)
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Markdown corpus file not found at: {DATA_PATH}")

#Embedding model used
#embeddings_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small"   #cheapest OpenAI embedding model
)

#Loading the Markdown document, Uses TextLoader to load the Markdown file content
loader = TextLoader(DATA_PATH)
raw_documents = loader.load()



## Recursive fixed length Chunking

In [4]:
#Splits the document using Recursive Character Split
#The fixed length is set at 700 characters per chunk with 100 characters overlap
#700 characters is about 112 words (115–140)
text_splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)
#Note: this method uses default separators built into LangChain: ["\n\n", "\n", " ", ""]
#Essentially splits based on fixed length but avoids splitting mid sentence, first splits by paragraphs, then sentences, then words.

#Applies the chunking to our baking corpus
chunks_fixed = text_splitter_recursive.split_documents(raw_documents)
#Print out how many documents:
print(f"Fixed-size chunking created {len(chunks_fixed)} chunks.")

#To create simple sequential chunk IDs (0 to N-1)
#This will be used later to retrieve chunks
# Assign metadata
for i, doc in enumerate(chunks_fixed):
    doc.metadata["chunk_id"] = i

# Save to Chroma (collection 1)
vector_store_fixed = Chroma.from_documents(
    documents=chunks_fixed,
    embedding=embeddings_model,
    collection_name="cooking_chunks_fixed",
    persist_directory=CHROMA_PATH,
    collection_metadata={"hnsw:space": "cosine"},
    ids=[str(i) for i in range(len(chunks_fixed))] #ensuring the chunk id is given in the metadata for later retrieval
)

#To be sure:
print("Chroma collection 'cooking_fixed' saved.")

Fixed-size chunking created 103 chunks.
Chroma collection 'cooking_fixed' saved.


In [5]:
#Testing it is using cosine similiarity 
collection = vector_store_fixed._collection
print("Collection metadata:", collection.metadata)


Collection metadata: {'hnsw:space': 'cosine'}


## Subheading Markdown Chunking

In [6]:
#Splits the document/markdown based on it's inherent structure 
#Will split Headings, subtitles, and subsubtitles
#To ensure it assoicates each subtitle to it's correct parent class
#The parent class has been repeated in the subtitle
#e.g. #Recipe for chocolate cake ##Ingredients ->Becomes: #Recipe for chocolate cake ##Ingredients chocolate cake, ect.

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "title"),
        ("##", "subtitle"),
        ("###", "subsubtitle")
    ]
)

#Breaks markdown file into different string chunks
chunks_markdown = markdown_splitter.split_text(raw_documents[0].page_content)

#See how many chunks created/test
print(f"Markdown chunking created {len(chunks_markdown)} chunks.")
#Assign chunk metadata (chunk_id)
for i, doc in enumerate(chunks_markdown):
    doc.metadata["chunk_id"] = i

#Save chunks into a Chroma collection
###################### collection two: cooking_chunks_markdown #################
vector_store_markdown = Chroma.from_documents(
    documents=chunks_markdown,
    embedding=embeddings_model,
    collection_name="cooking_chunks_markdown",
    persist_directory=CHROMA_PATH,
    collection_metadata={"hnsw:space": "cosine"},
    ids=[str(i) for i in range(len(chunks_markdown))]
)

print("Chroma collection 'cooking_markdown' saved.")

Markdown chunking created 58 chunks.
Chroma collection 'cooking_markdown' saved.


In [41]:
#Splits the document/markdown based on it's inherent structure 
#Will split Headings, subtitles, and subsubtitles
#To ensure it assoicates each subtitle to it's correct parent class
#The parent class has been repeated in the subtitle
#e.g. #Recipe for chocolate cake ##Ingredients ->Becomes: #Recipe for chocolate cake ##Ingredients chocolate cake, ect.

#To note, originally used MarkdownHeaderTextSplitter from langchain_text_splitters
#However, the issue is method automatically cuts out the headers and subheadings and I purposely repeated subheading and headings
#e.g. # straberry cake description  # straberry cake ingredients to allow retrievel
# Without the headings retrievel is poor
# Thus, my own custom heading and subheading chunking method:

#CUSTOM CHUNKER

def split_markdown_preserve_headers(text):
    """
    Splits markdown into chunks based on headings (#, ##, ###),
    and *keeps the heading inside each chunk*.
    """

    pattern = r'(?=^#{1,3} )'  
    #split before headings but keep them:
    ##^ means when there is a start of a line 
    ##?= means find the patten but don't consume/leave what you find/the header
    ##{1,3} Matches 1 to 3 hash signs: heading, subheading, and subsubeadings are grouped (although there were no subsubheadings in knowledgebase)
    raw_chunks = re.split(pattern, text, flags=re.MULTILINE)

    chunks = []
    for chunk in raw_chunks:
        chunk = chunk.strip()
        if chunk == "":
            continue
        chunks.append(Document(page_content=chunk))

    return chunks

#Getting the markdown content
markdown_text = raw_documents[0].page_content 
#Creating the chunks for baking markdown content
chunks_markdown = split_markdown_preserve_headers(markdown_text)

# To test chunking was correct
#First print how many chunsk created
print(f"Markdown chunking created {len(chunks_markdown)} chunks.")
#Create chunk ids
for i, doc in enumerate(chunks_markdown):
    doc.metadata["chunk_id"] = i
#testing to see how the chunks look (Are headings perserved)
chunk_id = 25  
print(chunks_markdown[chunk_id].page_content)

Markdown chunking created 65 chunks.
# Angel Food Cake Instructions
Instructions
Adjust the oven rack to the lower middle position and preheat oven to 325°F (163°C).

In a food processor or blender, pulse the sugar until fine and powdery. Remove 1 cup and set aside to use in step 3; keep the rest inside the food processor. Add the cake flour and salt to the food processor. Pulse 5-10 times until sugar/flour/salt mixture is aerated and light.
In a large bowl using a hand mixer or a stand mixer fitted with a whisk attachment, whip egg whites and cream of tartar together on medium-low until foamy, about 1 minute. Switch to medium-high and slowly add the 1 cup of sugar you set aside. Whip until soft peaks form, about 5-6 minutes. See photo and video above for a visual. Add the vanilla extract, then beat just until incorporated.
In 3 additions, slowly sift the flour mixture into the egg white mixture using a fine mesh strainer, gently folding with a rubber spatula after each addition. To av

In [47]:
#Adding these chunks to a new collection in our chroma database
###################### collection two: cooking_chunks_markdown #################
vector_store_markdown = Chroma.from_documents(
    documents=chunks_markdown,
    embedding=embeddings_model,
    collection_name="cooking_chunks_markdown",
    persist_directory=CHROMA_PATH,
    collection_metadata={"hnsw:space": "cosine"},
    ids=[str(i) for i in range(len(chunks_markdown))]
)

## Chunking Method 3: Semantic Chunking

In [7]:
#Semantic chunker uses embeddings (text-embedding-3-small")
semantic_chunker = SemanticChunker(embeddings_model)

#Splits markdown document based on semantic chunking
chunks_semantic_raw = semantic_chunker.split_text(raw_documents[0].page_content)

#to see how many chunks are created
print(f"Semantic chunking created {len(chunks_semantic_raw)} chunks.")

#Assign chunk IDs + convert into Document object
chunks_semantic = [
    Document(page_content=chunk, metadata={"chunk_id": i})
    for i, chunk in enumerate(chunks_semantic_raw)
]
#Save to Chroma database
###################### collection three: cooking_chunks_semantic #################
vector_store_semantic = Chroma.from_documents(
    documents=chunks_semantic,
    embedding=embeddings_model,
    collection_name="cooking_chunks_semantic",
    persist_directory=CHROMA_PATH,
    collection_metadata={"hnsw:space": "cosine"},
    ids=[str(i) for i in range(len(chunks_semantic))]
)
#to test
print("Chroma collection 'cooking_chunks_semantic' saved.")


Semantic chunking created 23 chunks.
Chroma collection 'cooking_chunks_semantic' saved.


In [8]:
#To test that all collections were created
from chromadb import HttpClient, PersistentClient  

client = PersistentClient(path=CHROMA_PATH)
print(client.list_collections())

col = client.get_collection("cooking_chunks_markdown")
print(col.count())

docs = vector_store_semantic.similarity_search("ingredients", k=1)
print(docs[0].metadata)

vec = embeddings_model.embed_query("test")
print(len(vec)) #1536-dimensional embedding space



[Collection(name=cooking_chunks_markdown), Collection(name=cooking_chunks_fixed), Collection(name=cooking_chunks_semantic)]
58
{'chunk_id': 22}
1536


In [51]:
#Testing all 3 collections were created
from chromadb import PersistentClient

client = PersistentClient(path="chroma_db")
collections = client.list_collections()

for c in collections:
    print(c.name)

cooking_chunks_markdown
cooking_chunks_fixed
cooking_chunks_semantic


In [53]:
#testing how many chunsk were created for each method

client = PersistentClient(path="chroma_db")
collections = client.list_collections()

for c in collections:
    col = client.get_collection(c.name)
    size = col.count()
    print(f"{c.name}: {size} documents")


cooking_chunks_markdown: 65 documents
cooking_chunks_fixed: 103 documents
cooking_chunks_semantic: 23 documents
